# Text Simplification Pipeline for ClearText using OpenRouter

In [1]:
import pandas as pd
import numpy as np
import os
from openai import OpenAI

In [2]:
# Change to each experiment ID for each inference
exp_id = "{ID}"

In [ ]:
# load OpenRouter API key through .env

from dotenv import load_dotenv
load_dotenv(os.path.expanduser("~/final_project/.env"))

In [4]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

### Data Processing

In [ ]:
#Loading in CLEARS dataset

# sampling paragraphs that have a max on 300 words
def load_clears_paired(csv_path, max_orig_words=300):
    df = pd.read_csv(csv_path)
    txt = df[df['type'] == 'TXT'][['row_id', 'text']].rename(columns={'text': 'original'})
    fac = df[df['type'] == 'FAC'][['row_id', 'text']].rename(columns={'text': 'simplified'})
    paired = txt.merge(fac, on='row_id')

    paired['orig_words'] = paired['original'].str.split().str.len()
    paired = paired[paired['orig_words'] <= max_orig_words].drop(columns='orig_words')
    return paired.reset_index(drop=True)

# Use the provided train/test splits
data_train = load_clears_paired('/home/c23068554/final_project/datasets/cleartext-discriminativo-es/train.csv')
data_test = load_clears_paired('/home/c23068554/final_project/datasets/cleartext-discriminativo-es/test.csv')

print(f"Train pairs (filtered): {len(data_train)}")
print(f"Test pairs (filtered): {len(data_test)}")

In [ ]:
# Sampling data for inference
from sklearn.model_selection import train_test_split

data_test, _ = train_test_split(data_test, train_size=134, random_state=59)
print(f"Subsampled to {len(data_test)} entries")

In [7]:
# selection for Few-shot count
random_state = 59
examples = data_train.sample(n=3, random_state = random_state)

### Prompts (Spanish)

In [8]:
BLESS_2 = "Por favor, reformula el siguiente texto complejo para que sea más comprensible para hablantes no nativos de español. Puedes hacerlo reemplazando palabras complejas por sinónimos más sencillos (parafraseando), eliminando información irrelevante (condensando) o dividiendo la oración en varias más simples. La oración simplificada final debe ser gramaticalmente correcta, fluida y conservar las ideas principales del original sin alterar su significado.\n\n"
PL_Guidelines = "Reescribe el siguiente párrafo según las Pautas Federales de Lenguaje Claro de EE. UU. para que sea más fácil de entender. 1. Usa la voz activa, no la pasiva. 2. Evita verbos implícitos (por ejemplo, 'tomar una decisión' se convierte en 'decidir'). 3. Usa palabras cortas, sencillas y comunes. 4. Evita la jerga y los términos técnicos. 5. Elimina las palabras innecesarias. 6. Usa oraciones cortas. 7. Limita cada oración a una sola idea. 8. Evita las dobles negaciones. El texto simplificado debe conservar el significado original.\n\n"
PA_Context = "Reescribe el siguiente párrafo sobre administración pública para que sea más fácil de entender. La complejidad del lenguaje administrativo se debe al uso excesivo de estructuras pasivas e impersonales, lenguaje formulista y vocabulario poco familiar. Los textos administrativos también contienen oraciones complejas, modificadores innecesarios y la voz pasiva. El texto simplificado debe conservar el significado original.\n\n"
PA_Target = "Reescribe el siguiente párrafo para que sea más comprensible para los ciudadanos que no son lectores expertos. Esto incluye a personas que no hablan español como lengua materna, personas con bajos niveles de alfabetización y adultos mayores. Estas personas no están familiarizadas con el lenguaje jurídico o administrativo y desean acceder a documentos públicos como ciudadanos. Pueden tener especial dificultad para comprender estructuras oracionales complejas, lenguaje burocrático y jerga legal. El texto simplificado debe conservar el significado original.\n\n"
PA_RoleBased = "Usted es un funcionario público responsable de mejorar la comunicación pública del gobierno. Su objetivo es reescribir documentos administrativos para que sean accesibles a todos los ciudadanos, incluyendo a aquellos con dominio limitado del español o sin formación jurídica. Reescriba el siguiente párrafo administrativo para que sea claro y fácil de entender para el público. El texto simplificado debe conservar el significado original.\n\n"

### Prompt (English)

In [9]:
BLESS_2_Eng = "Please rewrite the following complex text in order to make it easier to understand by non-native speakers of Spanish. You can do so by replacing complex words with simpler synonyms (i.e. paraphrasing), deleting unimportant information (i.e. compression), and/or splitting a long complex sentence into several simpler ones. The final simplified sentence needs to be grammatical, fluent, and retain the main ideas of its original counterpart without altering its meaning.\n\n"

### Prompt construction, instruction + few-shot examples


In [ ]:
instruction = "{SELECT PROMPT}"
def makePrompt(instruction, examples):
  #formatting text for fewshot examples
  fewshot = ""
  for index, row in examples.iterrows():
    fewshot += (f"Complex: {row.loc['original']}\nSimple: {row.loc['simplified']}\n\n")
  return(instruction + fewshot)

fewshot_example = makePrompt(instruction, examples)

## Load in model and inference

### Decoder models

In [11]:
# Decoder models for Spanish 
mistral_7b = "mistralai/mistral-7b-instruct-v0.1"
llama31_8b = "meta-llama/llama-3.1-8b-instruct"
gemma2_9b = "google/gemma-2-9b-it"
qwen2_7b = "qwen/qwen-2.5-7b-instruct"
llama33_70b = "meta-llama/llama-3.3-70b-instruct"

In [12]:
import time
model_id = qwen2_7b

def generate(prompt, max_tokens=600):
    for attempt in range(5):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": "Responda únicamente con el texto simplificado."},
                    {"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=1.0,
                top_p=0.9,
            )
            content = response.choices[0].message.content
            if content is None:
                print(f"  Empty response (attempt {attempt+1})")
                time.sleep(1)
                continue
            return content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e):
                wait = 2 ** attempt
                print(f"  Rate limited, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Max retries exceeded")


In [13]:
#looping through all n-shot prompt
def promptLoop(fewshot_example, data_test):
  LMsimplified = []
  for i, row in enumerate(data_test['original']):
    full_prompt = fewshot_example + f"Complex: {row}\nSimple:"
    LMsimplified.append(generate(full_prompt))
    #progress check on infference
    if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(data_test)} done")
  return LMsimplified


## Inference

In [ ]:
LMoutput = (promptLoop(fewshot_example, data_test))


In [16]:
# Clean prompt leakage from decoder outputs, using common LLM speech + Spanish terms
def clean_output(text):
    text = text.lstrip(':.,;>•*- \n')
    for delim in ['\nCompleja:', '\n\nCompleja:', '\nSimplificada:', '\n\nSimplificada:',
                  '\nComplex:', '\n\nComplex:', '\nNote:', '\n(Note:',
                  '\nNota:', '\nAnd also']:
        if delim in text:
            text = text.split(delim)[0]
    return text.strip()

LMoutput = [clean_output(s) for s in LMoutput]

### Organising outputs

In [17]:
reference = data_test['simplified'].tolist()
source = data_test['original'].tolist()

### Write Source, Reference and Prediction triplets to CSV

In [ ]:
import csv

out_path = "lm_output.csv"

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["Exp ID", "Index", "Original", "Reference", "LM Output"])
    for i, (orig, ref, lm) in enumerate(zip(source, reference, LMoutput)):
        writer.writerow([exp_id, i, orig, ref, lm])

print(f"Wrote {len(source)} rows to {out_path}")

# Evaluation

### Copy sentences count

In [ ]:
count = 0
for original, simple, lmSimple in zip(data_test['original'], data_test['simplified'], LMoutput):
  print(original)
  print(simple)
  print(lmSimple + "\n")
  if original == lmSimple:
    #print(original)
    #print(simple)
    #print(lmSimple)
    count += 1
print(f"Number of sentences not simplified or altered by the model: {count}")

### ROUGE, BLEU, BERTScore, SARI

In [ ]:
import evaluate

#load metrics
rouge = evaluate.load('rouge')
bleu = evaluate.load('bleu')
bertscore = evaluate.load('bertscore')
sari = evaluate.load('sari')

#Converting simplified reference sentences into a list inside a list
sari_references = [[s] for s in data_test["simplified"].astype(str).tolist()]
sari_score = sari.compute(sources= source, predictions= LMoutput, references= sari_references)

# Compute scores
rouge_results = rouge.compute(predictions= LMoutput, references = reference)
bleu_results = bleu.compute(predictions= LMoutput, references = reference)

# xlm-roberta-large is used as with Admin-It
bertscore_compute = bertscore.compute(predictions=LMoutput, references=reference, lang='es', model_type='xlm-roberta-large')
berstcoreAvg = np.mean(bertscore_compute['f1'])

### Fernandes Huerta Score

In [21]:
import textstat
textstat.set_lang("es")

fhOrig = np.mean([textstat.fernandez_huerta(s) for s in data_test["original"].tolist()])
fhSimp = np.mean([textstat.fernandez_huerta(s) for s in LMoutput])

In [ ]:
#output
print(f"Dataset: CLEARS (FAC), size: {len(data_test)}, random state: {random_state}")
print(f"Model: {model_id}")
print()
print(f"ROUGE Score: {rouge_results['rouge1']}")
print(f"BLEU Score: {bleu_results['bleu']}")
print(f"BERTScore Score (xlm-roberta-large): {berstcoreAvg}")
print(f"SARI Score: {sari_score['sari']}")
print()
print(f"Original Fernandez-Huerta: {fhOrig}")
print(f"Simplified Fernandez-Huerta: {fhSimp}")
print()
print(f"Copy count: {count}/{len(data_test)}")

In [ ]:
print(f"\nTAB-SEPARATED (paste into experiment sheet metrics columns):")
print(f"{sari_score['sari']:.4f}\t{rouge_results['rouge1']:.4f}\t{bleu_results['bleu']:.4f}\t{berstcoreAvg:.4f}\t{fhSimp:.4f}\t{count}/{len(data_test)}")

In [ ]:
#check disk space used:
# du -h --max-depth=1 ~ | sort -h

# command to clear cache often, to reduce disk space used:
# rm -rf ~/.cache/*

'''
Exporting CSV:
On local:
scp -r c23068554@10.98.84.2:/home/c23068554/final_project/lm_output.csv '/Users/justinwoodham/Desktop/CS/Y3/Final Year Project/raw outputs'
On Remote: 
rm lm_output.csv

'''